In [21]:

%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('Qt5Agg')  # Asegúrate de que este backend está instalado.
mne.viz.set_browser_backend('qt')  # Activa el backend interactivo

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

from neurodsp.spectral import compute_spectrum, trim_spectrum
from neurodsp.plts import plot_power_spectra

# Import IRASA related functions
from neurodsp.aperiodic import compute_irasa, fit_irasa


from joblib import Parallel, delayed


In [22]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_path         → g:\MOUS_204\MOUS_visual\output_source\source_block

In [35]:
f_min=1
f_max=50


#epochs
combinaciones = ["zinnen", "woorden"]


subjects=[]

# Recorremos cada subdirectorio en la carpeta base
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'
        
    if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
        # Añadimos el nombre del sujeto a la lista
        subjects.append(subdirectorio.name)

print(subjects)

##tablas de canales

channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]
channels_mag=channels_mag.tolist()
del channels

# epochs_zinnen=mne.read_epochs(epochs_clean_path / f"{subjects[0]}_epochs_zinnen_{layer_script}-epo.fif")
window_size=12
sliding_window=0.3

['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1017', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089'

In [ ]:
## obtener power spechtum de el objeto evoked -- esto serñía el MIXED POWER 

# psds=evoked.compute_psd(method='welch', fmin=fmin, fmax=fmax, reject_by_annotation=True, n_fft = int(6 * sfreq))
# psds_all.append(psds)
            
# psds,freqs= mne.time_frequency.psd_array_welch(array, sfreq, fmin=0, fmax=inf, n_fft=256, n_overlap=0, 
# #                                    n_per_seg=None, n_jobs=None, average='mean', window='hamming', remove_dc=True, *, 
# #                                    output='power', verbose=None)



##please note THAT THIS IS ONLY FOR EVOKED, YOU NEED TO CALCULATE IT THROUGH EPOCHS

# ##def compute_psd(self, fmin=0, fmax=np.inf, tmin=None, tmax=None, proj=False)
# array=evoked.get_data()
# sfreq=evoked.info['sfreq']
# fmin=0.5
# fmax=40
# nfft=1024 # to increase the spectral resolution, , although you can go to 4096 if you want to see more details
# njobs=10

# ##compute psd FOR WHOLE SIGNAL
# psds,freqs= mne.time_frequency.psd_array_welch(array, sfreq, fmin=fmin, fmax=fmax, n_fft=nfft, n_jobs=njobs)



NameError: name 'evoked' is not defined

In [79]:
subj="sub-V1001"
condition="zinnen"

path_epochs= epochs_clean_path / f"{subj}_epochs_{condition}_{layer_script}-epo.fif"
epochs = mne.read_epochs(path_epochs)


window_size=9
sliding_window=0.3
f_range=(4, 50)
hset=None 
thresh=None
isplot=False


##def dynamic_compute_ple(subj,epochs,condition ,window_size,sliding_window,f_range=(0.1, 50), hset=None, thresh=None, isplot=False):


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1001_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated


In [80]:





# def dynamic_compute_ple(subj,epochs,condition ,window_size,sliding_window,f_range=(0.5, 50), hset=None, thresh=None, isplot=False):
    # Filtrado de datos
    # Importa el objeto epochs y obtiene los datos SOLO DE EEG
# Importa el objeto epochs y obtiene los datos SOLO DE meg and exludes bad
if type(epochs)== mne.epochs.EpochsFIF:
    data_epochs=epochs.pick(picks="meg", exclude="bads").get_data()

#import epochs object and get data ONLY ON MEG DATA and ALSO EXCLUDE BADS

if type(epochs) == np.ndarray:
    data_epochs = epochs

#get duration
duration= epochs.tmax - epochs.tmin
#get sample frequency
sfreq= epochs.info['sfreq']
#get lags
lags=duration*sfreq

# Convertir a muestras
sliding_window_samples = int(sliding_window * sfreq)
window_size_samples = int(window_size * sfreq)

# ##ELECT ALL EPOCH ALL
psd_periodic_window_all_elect_all_epoch_all=[]
psd_aperiodic_window_all_elect_all_epoch_all=[]

slope_window_all_elect_all_epoch_all=[]
intercept_y_window_all_elect_all_epoch_all=[]

slope_slope_elect_all_epoch_all=[]
slope_std_elect_all_epoch_all=[]

intercept_y_slope_elect_all_epoch_all=[]
intercept_y_std_elect_all_epoch_all=[]

window_number = len(range(0, int(duration * sfreq) - window_size_samples + 1, sliding_window_samples))


#for each epoch
for i in range(0,len(data_epochs)):
    data_epoch=data_epochs[i]
    #print(f"una epoca tiene un shape de {data_epoch.shape}, con {data_epoch.shape[0]} canales y {data_epoch.shape[1]} puntos de tiempo")

    ##valores de ELECT_ALL en ONE EPOCH
    #values for window
    psd_aperiodic_window_all_elect_all_epoch=[]
    psd_periodic_window_all_elect_all_epoch=[]

    intercept_y_window_all_elect_all_epoch=[]
    slope_window_all_elect_all_epoch=[]

    #values for result
    slope_slope_elect_all_epoch=[]
    slope_std_elect_all_epoch=[]
    
    intercept_y_slope_elect_all_epoch=[]
    intercept_y_std_elect_all_epoch=[]


    #for each sensor on each epoch
    for h in range(0,len(data_epoch)):
        #en IRASA hay que meter el time series, es decir, el time series de un canal    
        epoch=data_epoch[h]
        n_samples = epoch.shape[0]
        ##valores de one ELECT en ONE EPOCH
        #values for windows
        psd_aperiodic_window_all_elect_epoch=[]
        psd_periodic_window_all_elect_epoch=[]

        intercept_y_window_all_elect_epoch=[]
        slope_window_all_elect_epoch=[]
        

        
        # Definir función respetando tu estructura
        def compute_irasa_fit(segment, sfreq, f_range, hset, thresh):
            freqs, psd_aperiodic_elect_epoch, psd_periodic_elect_epoch = compute_irasa(
                segment, fs=sfreq, f_range=f_range, hset=hset, thresh=thresh
            )
            intercept_y_window_elect_epoch, slope_elect_window_epoch = fit_irasa(
                freqs, psd_aperiodic_elect_epoch
            )
            return freqs,psd_aperiodic_elect_epoch, psd_periodic_elect_epoch, intercept_y_window_elect_epoch, slope_elect_window_epoch

#

        # Crear lista de segmentos válidos
        segments = [
            epoch[start:start + window_size_samples]
            for start in range(0, n_samples - window_size_samples + 1, sliding_window_samples)
            if epoch[start:start + window_size_samples].shape[0] == window_size_samples
        ]

        
        # Procesar en paralelo con joblib
        window_results = Parallel(n_jobs=-1)(
            delayed(compute_irasa_fit)(segment, sfreq, f_range, hset, thresh)
            for segment in segments
        )

        # Desempaquetar resultados
        for freqs,psd_aperiodic_elect_epoch, psd_periodic_elect_epoch, intercept_y_window_elect_epoch, slope_elect_window_epoch in window_results:
            freqs_def=freqs
            psd_aperiodic_window_all_elect_epoch.append(psd_aperiodic_elect_epoch)
            psd_periodic_window_all_elect_epoch.append(psd_periodic_elect_epoch)
            intercept_y_window_all_elect_epoch.append(intercept_y_window_elect_epoch)
            slope_window_all_elect_epoch.append(slope_elect_window_epoch)
                
        ##append values for ELECT_ALL in ONE EPOCH
        psd_aperiodic_window_all_elect_all_epoch.append(psd_aperiodic_window_all_elect_epoch)
        psd_periodic_window_all_elect_all_epoch.append(psd_periodic_window_all_elect_epoch)

        intercept_y_window_all_elect_all_epoch.append(intercept_y_window_all_elect_epoch)
        slope_window_all_elect_all_epoch.append(slope_window_all_elect_epoch)
        
        
        
        ##generate and apend values for results FOR ELECT_ALL in ONE EPOCH
        x=range(0,len(slope_window_all_elect_epoch))
        y=slope_window_all_elect_epoch
        coeffs_slope=np.polyfit(x,y, deg=1)
        slope_slope = coeffs_slope[0]
        slope_std = np.std(slope_window_all_elect_epoch)
        slope_slope_elect_all_epoch.append(slope_slope)
        slope_std_elect_all_epoch.append(slope_std)
        
        x=range(0,len(intercept_y_window_all_elect_epoch))
        y=intercept_y_window_all_elect_epoch
        coeffs_intercept_y=np.polyfit(x,y, deg=1)
        intercept_y_slope = coeffs_intercept_y[0]
        intercept_y_std = np.std(intercept_y_window_all_elect_epoch)
        intercept_y_slope_elect_all_epoch.append(intercept_y_slope)
        intercept_y_std_elect_all_epoch.append(intercept_y_std)


        
    ##apend values for ELECT_ALL in EPOCH_all
    psd_periodic_window_all_elect_all_epoch_all.append(psd_periodic_window_all_elect_all_epoch)
    psd_aperiodic_window_all_elect_all_epoch_all.append(psd_aperiodic_window_all_elect_all_epoch)
    
    slope_window_all_elect_all_epoch_all.append(slope_window_all_elect_all_epoch)
    intercept_y_window_all_elect_all_epoch_all.append(intercept_y_window_all_elect_all_epoch)
    
    slope_slope_elect_all_epoch_all.append(slope_slope_elect_all_epoch)
    slope_std_elect_all_epoch_all.append(slope_std_elect_all_epoch)
    
    intercept_y_slope_elect_all_epoch_all.append(intercept_y_slope_elect_all_epoch)
    intercept_y_std_elect_all_epoch_all.append(intercept_y_std_elect_all_epoch)


# ##promedio de todos los valores de los electrodos en todas las epocas
#     ##conversiñón en arrays
# psd_periodict_elect_all_epoch_all_array=np.array(psd_periodict_elect_all_epoch_all)
# psd_aperiodic_elect_all_epoch_all_array=np.array(psd_aperiodic_elect_all_epoch_all)

# intercept_y_elect_all_epoch_all_array=np.array(intercept_y_elect_all_epoch_all)
# slope_elect_all_epoch_all_array=np.array(slope_elect_all_epoch_all)

# ##computo de la media de TODOS los electrodos en CADA EPOCA
# psd_periodict_elect_mean_epoch_all=np.mean(psd_periodict_elect_all_epoch_all_array, axis=1)
# psd_aperiodic_elect_mean_epoch_all=np.mean(psd_aperiodic_elect_all_epoch_all_array, axis=1)

# intercept_y_elect_mean_epoch_all=np.mean(intercept_y_elect_all_epoch_all_array, axis=1)
# slope_elect_mean_epoch_all=np.mean(slope_elect_all_epoch_all_array, axis=1)


# ##computo de la media de CADA los electrodos en TODAS EPOCAS
# psd_periodict_elect_all_epoch_mean=np.mean(psd_periodict_elect_all_epoch_all_array, axis=0)
# psd_aperiodic_elect_all_epoch_mean=np.mean(psd_aperiodic_elect_all_epoch_all_array, axis=0)

# intercept_y_elect_all_epoch_mean=np.mean(intercept_y_elect_all_epoch_all_array, axis=0)
# slope_elect_all_epoch_mean=np.mean(slope_elect_all_epoch_all_array, axis=0)



#parameters of table
num_elects=len(channels_mag)
num_epochs=len(data_epochs)
shape_tabla_dynamic=num_epochs*num_elects*window_number
shape_tabla_results=num_epochs*num_elects




# psd_periodic_elect_all_epoch_all_list = np.array(psd_periodic_elect_all_epoch_all).reshape(num_epochs * num_elects, np.array(psd_periodic_elect_all_epoch_all).shape[2]).tolist()
# psd_aperiodic_elect_all_epoch_all_list = np.array(psd_aperiodic_elect_all_epoch_all).reshape(num_epochs * num_elects, np.array(psd_aperiodic_elect_all_epoch_all).shape[2]).tolist()



table_dynamic_PLE = pd.DataFrame({
    'Subject': [subj] * shape_tabla_dynamic,  # Repite el sujeto para todas las filas
    'Condition': [condition] * shape_tabla_dynamic,  # Repite la condición para todas las filas
    "freqs":[freqs_def]*shape_tabla_dynamic,
    'Epoch': np.repeat(np.arange(num_epochs), num_elects * window_number),
    'Elect': np.tile(np.repeat(channels_mag, window_number), num_epochs),
    'Window': np.tile(np.arange(window_number), num_epochs * num_elects),
    # 'psd_periodic_elect_all_epoch_all': psd_periodic_elect_all_epoch_all_list,
    # "psd_aperiodic_elect_all_epoch_all": psd_aperiodic_elect_all_epoch_all_list,
    'intercept_y_window_all_elect_all_epoch_all': np.array(intercept_y_window_all_elect_all_epoch_all).flatten(),
    'slope_window_all_elect_all_epoch_all': np.array(slope_window_all_elect_all_epoch_all).flatten(),
    # 'psd_periodict_elect_mean_epoch_all': [psd_periodict_elect_mean_epoch_all] * len(psd_periodict_elect_all_epoch_all),
    # 'psd_aperiodic_elect_mean_epoch_all': [psd_aperiodic_elect_mean_epoch_all] * len(psd_periodict_elect_all_epoch_all),
    # "intercept_y_elect_mean_epoch_all": [intercept_y_elect_mean_epoch_all] * len(psd_periodict_elect_all_epoch_all),
    # "slope_elect_mean_epoch_all": [slope_elect_mean_epoch_all] * len(psd_periodict_elect_all_epoch_all),
    # 'psd_periodict_elect_all_epoch_mean': [psd_periodict_elect_all_epoch_mean] * len(psd_periodict_elect_all_epoch_all),
    # 'psd_aperiodic_elect_all_epoch_mean': [psd_aperiodic_elect_all_epoch_mean] * len(psd_periodict_elect_all_epoch_all),
    # "intercept_y_elect_all_epoch_mean": [intercept_y_elect_all_epoch_mean] * len(psd_periodict_elect_all_epoch_all),
    # "slope_elect_all_epoch_mean": [slope_elect_all_epoch_mean] * len(psd_periodict_elect_all_epoch_all)
    })

table_dynamic_PLE_results = pd.DataFrame({
    'Subject': [subj] * shape_tabla_results,
    'Condition': [condition] * shape_tabla_results,
    'Epoch': np.repeat(np.arange(num_epochs), num_elects),
    'Elect': np.tile(channels_mag, num_epochs),
    # 'psd_periodic_elect_all_epoch_all': psd_periodic_elect_all_epoch_all_list,
    # "psd_aperiodic_elect_all_epoch_all": psd_aperiodic_elect_all_epoch_all_list,
    'slope_slope_elect_all_epoch_all': np.array(slope_slope_elect_all_epoch_all).flatten(),
    "slope_std_elect_all_epoch_all": np.array(slope_std_elect_all_epoch_all).flatten(),
    'intercept_y_slope_elect_all_epoch_all': np.array(intercept_y_slope_elect_all_epoch_all).flatten(),
    "intercept_y_std_elect_all_epoch_all": np.array(intercept_y_std_elect_all_epoch_all).flatten(),
    # 'psd_periodict_elect_mean_epoch_all': [psd_periodict_elect_mean_epoch_all] * len(psd_periodict_elect_all_epoch_all),
    # 'psd_aperiodic_elect_mean_epoch_all': [psd_aperiodic_elect_mean_epoch_all] * len(psd_periodict_elect_all_epoch_all),
    # "intercept_y_elect_mean_epoch_all": [intercept_y_elect_mean_epoch_all] * len(psd_periodict_elect_all_epoch_all),
    # "slope_elect_mean_epoch_all": [slope_elect_mean_epoch_all] * len(psd_periodict_elect_all_epoch_all),
    # 'psd_periodict_elect_all_epoch_mean': [psd_periodict_elect_all_epoch_mean] * len(psd_periodict_elect_all_epoch_all),
    # 'psd_aperiodic_elect_all_epoch_mean': [psd_aperiodic_elect_all_epoch_mean] * len(psd_periodict_elect_all_epoch_all),
    # "intercept_y_elect_all_epoch_mean": [intercept_y_elect_all_epoch_mean] * len(psd_periodict_elect_all_epoch_all),
    # "slope_elect_all_epoch_mean": [slope_elect_all_epoch_mean] * len(psd_periodict_elect_all_epoch_all)


    })
    
    
    
    # return table_dynamic_PLE, table_dynamic_PLE_results



C:\Users\UCM\AppData\Local\Temp\ipykernel_1084\2868484050.py:6: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.pick(picks="meg", exclude="bads").get_data()


In [39]:
table_dynamic_PLE, table_dynamic_PLE_results=dynamic_compute_ple(subj,epochs,combinacion,window_size, sliding_window,  f_range=(f_min, f_max),isplot=False)

C:\Users\UCM\AppData\Local\Temp\ipykernel_1084\325650543.py:6: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.pick(picks="meg", exclude="bads").get_data()


ValueError: All arrays must be of the same length

In [29]:
segment

NameError: name 'segment' is not defined

In [ ]:
#table_PLE
#del table_PLE

,Subject,Condition,freqs,Epoch,Elect,psd_periodic_elect_all_epoch_all,psd_aperiodic_elect_all_epoch_all,intercept_y_elect_all_epoch_all,slope_elect_all_epoch_all
0,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,MLC11-4304,"[6.388444458053898e-28, 2.680178974171939e-28,...","[1.5960253454463609e-27, 3.209103275500903e-27...",-26.590514,-1.110627
1,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,MLC12-4304,"[5.3480790742951e-28, 2.292580777795326e-28, 1...","[2.3825742762335104e-27, 4.711772708421555e-27...",-26.456013,-1.151594
2,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,MLC13-4304,"[3.2710181502307626e-28, 1.0439785835775173e-2...","[3.087726793168582e-27, 5.736748103642964e-27,...",-26.319605,-1.258218
3,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,MLC14-4304,"[2.1017081495394065e-28, 1.2187840606717464e-2...","[3.57464840185941e-27, 7.062962040379582e-27, ...",-26.186353,-1.328133
4,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,MLC15-4304,"[-4.459980860415077e-29, 1.0067953221739206e-2...","[4.668801673597025e-27, 8.428644379826931e-27,...",-26.114707,-1.309358
...,...,...,...,...,...,...,...,...,...
6547,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",23,MZF03-4304,"[3.3648455099897247e-28, 6.798568959065484e-28...","[8.125591187857677e-28, 1.399413892716638e-27,...",-26.949352,-0.972082
6548,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",23,MZO01-4304,"[3.2968889195629917e-28, -2.121286482407277e-2...","[2.237531420507963e-27, 3.303688151340406e-27,...",-26.295425,-1.015202
6549,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",23,MZO02-4304,"[6.612820204451016e-28, -2.0516916606295286e-2...","[3.03354805260208e-27, 4.289975622698823e-27, ...",-26.439506,-0.440536
6550,sub-V1001,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",23,MZO03-4304,"[5.207639985049704e-28, 1.6769514867088223e-28...","[5.1290586189815666e-27, 5.693450240919498e-27...",-26.648812,-0.020083


In [ ]:
subject=subj[i]
combinacion= combinaciones[h]
path_epochs= epochs_clean_path / f"{subject}_epochs_{combinacion}_{layer_script}-epo.fif"
epochs = mne.read_epochs(path_epochs)
table_PLE=compute_ple(subject,epochs, condition=combinacion, f_range=(f_min, f_max),isplot=False)

NameError: name 'table_PLE' is not defined

In [15]:
##codigo para agrupar todas las tablas

all_tables = []

for i in range(0,len(subj)):

    for h in range(0,len(combinaciones)):
        try:
            subject=subj[i]
            combinacion= combinaciones[h]
            path_epochs= epochs_clean_path / f"{subject}_epochs_{combinacion}_{layer_script}-epo.fif"
            epochs = mne.read_epochs(path_epochs)
            table_PLE=compute_ple(subject,epochs, condition=combinacion, f_range=(f_min, f_max),isplot=False)
            all_tables.append(table_PLE)
            del epochs
        except:
            print(f"Error en {subject} en {combinacion}")
            continue

table_PLE_subjects_all = pd.concat(all_tables, ignore_index=True)

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1001_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_740\948296545.py:5: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1001_epochs_woorden_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
20 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_740\948296545.py:5: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1002_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
22 matching events found
No baseline correction applied
0 projection items activated


...

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1024_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_740\948296545.py:5: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1024_epochs_woorden_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_740\948296545.py:5: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs=epochs.copy().pick(picks="meg", exclude="bads").get_data()


Error en sub-V1025 en zinnen
Error en sub-V1025 en woorden
Error en sub-V1026 en zinnen
Error en sub-V1026 en woorden
Error en sub-V1027 en zinnen
Error en sub-V1027 en woorden
Error en sub-V1028 en zinnen
Error en sub-V1028 en woorden
Error en sub-V1029 en zinnen
Error en sub-V1029 en woorden
Error en sub-V1030 en zinnen
Error en sub-V1030 en woorden
Error en sub-V1031 en zinnen
Error en sub-V1031 en woorden
Error en sub-V1032 en zinnen
Error en sub-V1032 en woorden
Error en sub-V1033 en zinnen
Error en sub-V1033 en woorden
Error en sub-V1034 en zinnen
Error en sub-V1034 en woorden
Error en sub-V1035 en zinnen
Error en sub-V1035 en woorden
Error en sub-V1036 en zinnen
Error en sub-V1036 en woorden
Error en sub-V1037 en zinnen
Error en sub-V1037 en woorden
Error en sub-V1038 en zinnen
Error en sub-V1038 en woorden
Error en sub-V1039 en zinnen
Error en sub-V1039 en woorden
Error en sub-V1040 en zinnen
Error en sub-V1040 en woorden
Error en sub-V1042 en zinnen
Error en sub-V1042 en woord

In [15]:
table_PLE_subjects_all

,Subject,Condition,freqs,Epoch,Elect,psd_periodic_elect_all_epoch_all,psd_aperiodic_elect_all_epoch_all,intercept_y_elect_all_epoch_all,slope_elect_all_epoch_all
0,sub-A2002,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,0,"[8.674628326852714e-29, 2.654025077790484e-29,...","[1.5690735244975993e-27, 2.5321316036926838e-2...",-25.729772,-2.154004
1,sub-A2002,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,1,"[-1.078348235138371e-28, 5.331220784760329e-28...","[2.4140977474362464e-27, 3.965004780839369e-27...",-25.553466,-2.229709
2,sub-A2002,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,2,"[-2.257588900963778e-30, 1.234604715733112e-27...","[3.473149622122688e-27, 5.708915014485239e-27,...",-25.403916,-2.328152
3,sub-A2002,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,3,"[3.422320523073629e-28, 2.792819339291622e-27,...","[4.6134061287522916e-27, 7.101616286264271e-27...",-25.288044,-2.382093
4,sub-A2002,zinnen,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",0,4,"[9.637138301484e-28, 4.257668333535582e-27, 2....","[4.5108211928352536e-27, 7.519187012228288e-27...",-25.208322,-2.419240
...,...,...,...,...,...,...,...,...,...
224445,sub-A2025,woorden,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",23,268,"[-4.25212816462578e-28, -1.249561013869523e-27...","[2.90317872581447e-27, 4.8994364347754585e-27,...",-25.521998,-2.250522
224446,sub-A2025,woorden,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",23,269,"[2.323411003852139e-28, -1.2848510529921378e-2...","[2.472292952772411e-27, 4.135445263201128e-27,...",-25.054620,-2.463384
224447,sub-A2025,woorden,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",23,270,"[1.1320515801451662e-28, -4.12794347956715e-28...","[1.081422369089005e-27, 1.6806180124960702e-27...",-25.489018,-2.282894
224448,sub-A2025,woorden,"[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2...",23,271,"[-6.811681059148415e-30, -2.7062440458550817e-...","[2.403461230311065e-28, 4.241425607809252e-28,...",-26.201930,-2.074193


In [16]:
PLE_path


WindowsPath('g:/MOUS_204/MOUS_visual/output_analysis/analysis_block/PLE_block')

In [17]:
table_PLE_subjects_all.to_pickle(PLE_path / f"table_PLE_subjects_all.pickle")